# P21 – Tumour Tagger
### Breast Cancer Classification using Logistic Regression

- **Student:** Rushan Shaikh
- **Roll No.:** H39
- **Division:** E
- **Enrollment No.:** 24004501210400
- **Project ID:** P21

**Objective:** Use cell-nucleus measurements to classify a tumour diagnosis as benign (B) or malignant (M). This is an educational machine-learning project, not a medical diagnostic tool.

## Step 1 — Import libraries

This cell imports the libraries used for data handling, visualisation, preprocessing, feature selection, Logistic Regression, and evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay
)

print("All libraries loaded!")

## Step 2 — Load the dataset

Keep `data.csv` in the same folder as this notebook. If your downloaded file has a different name, change `DATA_FILE` below.

In [ ]:
from pathlib import Path

# Support the filename in the project guide and the filename used in your current notebook.
possible_files = ["data.csv", "ml_data_p21.csv"]
DATA_FILE = next((name for name in possible_files if Path(name).exists()), None)

if DATA_FILE is None:
    raise FileNotFoundError(
        "Dataset not found. Put data.csv or ml_data_p21.csv in the same folder as this notebook."
    )

print("Loading dataset:", DATA_FILE)
df = pd.read_csv(DATA_FILE)
print("Dataset shape (rows, columns):", df.shape)
display(df.head())

## Step 3 — Understand the dataset

Review column types, summary statistics, missing values, and duplicate rows before cleaning.

In [ ]:
df.info()

In [ ]:
display(df.describe().T)

In [ ]:
missing_before = df.isnull().sum().sort_values(ascending=False)
display(missing_before[missing_before > 0])
print("Duplicate rows:", df.duplicated().sum())

## Step 4 — Check the target column

The target is `diagnosis`. The dataset labels are B (benign) and M (malignant). Class counts help us understand the target distribution.

In [ ]:
TARGET = "diagnosis"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' was not found. Check the CSV file and column names.")

display(df[TARGET].value_counts().rename_axis("Diagnosis").to_frame("Count"))
display((df[TARGET].value_counts(normalize=True) * 100).round(2).rename("Percentage"))

## Step 5 — Clean the data

Remove duplicate rows and identifier/empty columns. Fill missing numerical feature values with the median and categorical feature values with the most frequent value. Rows without a target are removed.

In [ ]:
df = df.drop_duplicates().copy()

# Remove identifier and empty columns when present.
columns_to_drop = [col for col in ["id", "Unnamed: 32"] if col in df.columns]
df = df.drop(columns=columns_to_drop)

# Remove rows where the target is missing.
df = df.dropna(subset=[TARGET]).copy()

# Fill missing values in input columns.
for col in df.columns:
    if col == TARGET:
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        if df[col].notna().any():
            df[col] = df[col].fillna(df[col].median())
    else:
        mode_values = df[col].mode(dropna=True)
        if len(mode_values) > 0:
            df[col] = df[col].fillna(mode_values.iloc[0])

print("Columns removed:", columns_to_drop)
print("Cleaned dataset shape:", df.shape)
print("Missing values remaining:", int(df.isnull().sum().sum()))

## Step 6 — Create a data dictionary

This table describes the dataset columns, data types, and whether each column is used for modelling. Meanings are assigned from common column names in the Wisconsin Breast Cancer dataset; confirm these against your dataset if column names differ.

In [ ]:
def describe_column(col):
    if col == TARGET:
        return "Target label: benign (B) or malignant (M)"
    if col == "id":
        return "Record identifier; not a predictive measurement"
    if col == "Unnamed: 32":
        return "Empty extra column; remove"
    if col.endswith("_mean"):
        return "Mean value of a cell-nucleus measurement"
    if col.endswith("_se"):
        return "Standard error of a cell-nucleus measurement"
    if col.endswith("_worst"):
        return "Worst/largest value of a cell-nucleus measurement"
    return "Cell-nucleus measurement or derived feature"

dictionary_rows = []
for col in df.columns:
    dictionary_rows.append({
        "Column name": col,
        "Meaning": describe_column(col),
        "Type": "Text" if not pd.api.types.is_numeric_dtype(df[col]) else "Number",
        "Use": "Target (Y)" if col == TARGET else "Input feature (X)"
    })

data_dictionary = pd.DataFrame(dictionary_rows)
display(data_dictionary)

## Step 7 — Exploratory Data Analysis (EDA)

The charts below show class balance, feature distributions, differences between diagnoses, and correlations among numeric measurements.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET)
plt.title("Diagnosis Class Balance")
plt.xlabel("Diagnosis (B = Benign, M = Malignant)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols_for_eda = df.select_dtypes(include="number").columns.tolist()
df[numeric_cols_for_eda].hist(figsize=(14, 10), bins=25)
plt.suptitle("Distributions of Numeric Features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare a commonly used measurement by diagnosis when available.
box_feature = "radius_mean" if "radius_mean" in df.columns else numeric_cols_for_eda[0]
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x=TARGET, y=box_feature)
plt.title(f"{box_feature} by Diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df[numeric_cols_for_eda].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()

## Step 8 — Record five EDA insights

Use the charts above to verify these observations. The statements below are prompts based on common patterns in this dataset; edit them if your own charts show something different.

1. The diagnosis count plot shows that benign cases are more numerous than malignant cases in this dataset.
2. Feature histograms show that numerical measurements have different ranges and distributions.
3. The box plot lets us compare the selected measurement between benign and malignant cases.
4. The correlation heatmap shows that some size-related measurements are strongly correlated with one another.
5. Highly correlated measurements may contain overlapping information, which motivates feature selection before model training.

## Step 9 — Create a new feature

Create a ratio from existing measurements. The area-to-perimeter ratio is a derived feature that combines two size measurements. It is used for exploration and may or may not be selected by the feature-selection step.

In [ ]:
if {"area_mean", "perimeter_mean"}.issubset(df.columns):
    df["area_perimeter_ratio_mean"] = df["area_mean"] / df["perimeter_mean"].replace(0, np.nan)
    df["area_perimeter_ratio_mean"] = df["area_perimeter_ratio_mean"].replace([np.inf, -np.inf], np.nan)
    df["area_perimeter_ratio_mean"] = df["area_perimeter_ratio_mean"].fillna(
        df["area_perimeter_ratio_mean"].median()
    )
    print("Created feature: area_perimeter_ratio_mean")
else:
    print("The expected area_mean/perimeter_mean columns were not found; no derived feature was created.")
    print("Check the dataset column names before continuing.")

## Step 10 — Separate inputs and target

`X` contains the input features and `y` contains the diagnosis label. We map benign to 0 and malignant to 1 so recall focuses on identifying malignant cases.

In [ ]:
X = df.drop(columns=[TARGET]).copy()
y = (df[TARGET].astype(str).str.strip().str.upper() == "M").astype(int)

print("Target mapping: B = 0, M = 1")
print("Class counts (0 = B, 1 = M):")
print(y.value_counts().sort_index())
print("Input shape before preprocessing:", X.shape)

## Step 11 — Split data into training and testing sets

Split before fitting the imputer, feature selector, or scaler. This helps prevent information from the test set leaking into training.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", X_train_raw.shape[0])
print("Testing rows:", X_test_raw.shape[0])

## Step 12 — Impute missing values and encode input columns

Fit the imputer using training data only. Then apply the same imputer to the test data. Convert any categorical input columns into numeric indicator columns.

In [ ]:
# Drop any remaining completely empty input columns using training data only.
empty_train_cols = X_train_raw.columns[X_train_raw.isnull().all()].tolist()
X_train_raw = X_train_raw.drop(columns=empty_train_cols)
X_test_raw = X_test_raw.drop(columns=empty_train_cols, errors="ignore")

# One-hot encode categorical columns consistently across train and test.
X_train_encoded = pd.get_dummies(X_train_raw, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Impute any missing numeric values based on the training set.
imputer = SimpleImputer(strategy="median", keep_empty_features=True)
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train_encoded),
    columns=X_train_encoded.columns,
    index=X_train_encoded.index
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test_encoded),
    columns=X_train_encoded.columns,
    index=X_test_encoded.index
)

print("Training inputs after preprocessing:", X_train_imputed.shape)
print("Testing inputs after preprocessing:", X_test_imputed.shape)
print("Missing values in training inputs:", int(X_train_imputed.isnull().sum().sum()))
print("Missing values in testing inputs:", int(X_test_imputed.isnull().sum().sum()))

## Step 13 — Select the 15 most useful features

`SelectKBest` ranks features using the ANOVA F-test. Fit it on training data only, then apply it to the test data. This also avoids selecting features using the test labels.

In [ ]:
k = min(15, X_train_imputed.shape[1])
selector = SelectKBest(score_func=f_classif, k=k)
X_train_selected_array = selector.fit_transform(X_train_imputed, y_train)
X_test_selected_array = selector.transform(X_test_imputed)

feature_names = X_train_imputed.columns[selector.get_support()].tolist()
X_train_selected = pd.DataFrame(
    X_train_selected_array, columns=feature_names, index=X_train_imputed.index
)
X_test_selected = pd.DataFrame(
    X_test_selected_array, columns=feature_names, index=X_test_imputed.index
)

feature_scores = pd.Series(selector.scores_, index=X_train_imputed.columns)
feature_scores = feature_scores.sort_values(ascending=False)

print("Selected features:")
for name in feature_names:
    print("-", name)
display(feature_scores.head(20).to_frame("ANOVA F-score"))

## Step 14 — Scale features

StandardScaler learns the mean and standard deviation from the training data only. The fitted scaler is then applied to the test data.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_selected)
X_test = scaler.transform(X_test_selected)

print("Scaled training shape:", X_train.shape)
print("Scaled testing shape:", X_test.shape)

## Step 15 — Define a model evaluation function

All model settings use Logistic Regression. The evaluation function reports accuracy, precision, malignant recall, F1-score, specificity, and ROC-AUC.

In [ ]:
results = []

def evaluate(name, model, show=False):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_probability = model.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    row = {
        "Setting": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Malignant Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Specificity": specificity,
        "AUC": roc_auc_score(y_test, y_probability)
    }

    if show:
        display(pd.DataFrame(cm, index=["Actual B", "Actual M"], columns=["Predicted B", "Predicted M"]))
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["B", "M"]).plot(cmap="Blues")
        plt.title(name + " — Confusion Matrix")
        plt.tight_layout()
        plt.show()

        print(classification_report(
            y_test, y_pred, labels=[0, 1], target_names=["Benign (B)", "Malignant (M)"],
            zero_division=0
        ))
        RocCurveDisplay.from_predictions(y_test, y_probability)
        plt.title(name + " — ROC Curve")
        plt.tight_layout()
        plt.show()

    results.append(row)
    print(row)
    return row

## Step 16 — Train the baseline Logistic Regression model

The baseline uses Logistic Regression with `max_iter=1000`. Record these metrics as the reference point for comparing later settings.

In [ ]:
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_result = evaluate("Baseline (default class weight)", baseline_model, show=True)

## Step 17 — Compare at least four Logistic Regression settings

Compare different regularisation strengths and class weighting. These are all Logistic Regression models, as required by the project guide.

In [ ]:
settings = [
    ("class_weight=None, C=0.1", LogisticRegression(max_iter=1000, C=0.1, class_weight=None, random_state=42)),
    ("class_weight=None, C=1.0", LogisticRegression(max_iter=1000, C=1.0, class_weight=None, random_state=42)),
    ("class_weight=balanced, C=1.0", LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=42)),
    ("class_weight=balanced, C=10.0", LogisticRegression(max_iter=1000, C=10.0, class_weight="balanced", random_state=42))
]

for setting_name, setting_model in settings:
    evaluate(setting_name, setting_model, show=False)

## Step 18 — Inspect model coefficients

Coefficients show how each selected feature relates to the log-odds of the malignant class, holding the other selected features constant. Because features are scaled, their coefficient magnitudes are more comparable.

In [ ]:
coefficient_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
coefficient_model.fit(X_train, y_train)

coefficients = pd.Series(coefficient_model.coef_[0], index=feature_names).sort_values()
display(coefficients.to_frame("Coefficient"))

coefficients.plot(kind="barh", figsize=(9, 7))
plt.title("Logistic Regression Coefficients (Balanced Model)")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()

## Step 19 — Test different probability thresholds

A lower threshold generally identifies more malignant cases (higher recall), but may also increase false positives. This cell reports precision and recall at thresholds 0.3, 0.5, and 0.7.

In [ ]:
threshold_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
threshold_model.fit(X_train, y_train)
malignant_probability = threshold_model.predict_proba(X_test)[:, 1]

threshold_results = []
for threshold in [0.3, 0.5, 0.7]:
    threshold_prediction = (malignant_probability >= threshold).astype(int)
    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, threshold_prediction, pos_label=1, zero_division=0),
        "Malignant Recall": recall_score(y_test, threshold_prediction, pos_label=1, zero_division=0)
    })

threshold_table = pd.DataFrame(threshold_results).round(3)
display(threshold_table)

## Step 20 — Compare model results and choose the final setting

The comparison table contains the baseline and all tested settings. The project guide prioritises malignant recall and AUC. This cell selects the highest-recall setting, using AUC as a tie-breaker.

In [ ]:
results_df = pd.DataFrame(results).drop_duplicates(subset=["Setting"], keep="last")
results_df = results_df.sort_values(
    by=["Malignant Recall", "AUC"], ascending=[False, False]
).reset_index(drop=True)

display(results_df.round(3))

results_df.set_index("Setting")[["Accuracy", "Precision", "Malignant Recall", "F1-score", "AUC"]].plot(
    kind="bar", figsize=(12, 5)
)
plt.title("Comparison of Logistic Regression Settings")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

best_setting_name = results_df.iloc[0]["Setting"]
print("Best setting by malignant recall, with AUC as tie-breaker:", best_setting_name)

## Step 21 — Evaluate the final model

Refit the selected Logistic Regression setting and display its final confusion matrix, classification report, ROC curve, and metrics.

In [ ]:
model_lookup = {name: model for name, model in settings}
if best_setting_name == "Baseline (default class weight)":
    final_model = LogisticRegression(max_iter=1000, random_state=42)
else:
    final_model = model_lookup[best_setting_name]

# Fit and report final metrics without adding a duplicate row to the comparison table.
final_model.fit(X_train, y_train)
final_pred = final_model.predict(X_test)
final_probability = final_model.predict_proba(X_test)[:, 1]
final_cm = confusion_matrix(y_test, final_pred, labels=[0, 1])
tn, fp, fn, tp = final_cm.ravel()

final_metrics = {
    "Accuracy": accuracy_score(y_test, final_pred),
    "Precision (malignant)": precision_score(y_test, final_pred, pos_label=1, zero_division=0),
    "Malignant Recall": recall_score(y_test, final_pred, pos_label=1, zero_division=0),
    "F1-score (malignant)": f1_score(y_test, final_pred, pos_label=1, zero_division=0),
    "Specificity (benign)": tn / (tn + fp) if (tn + fp) else 0.0,
    "AUC": roc_auc_score(y_test, final_probability)
}
display(pd.DataFrame([final_metrics]).round(4))

ConfusionMatrixDisplay(confusion_matrix=final_cm, display_labels=["B", "M"]).plot(cmap="Blues")
plt.title("Final Model — Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(
    y_test, final_pred, labels=[0, 1], target_names=["Benign (B)", "Malignant (M)"],
    zero_division=0
))
RocCurveDisplay.from_predictions(y_test, final_probability)
plt.title("Final Model — ROC Curve")
plt.tight_layout()
plt.show()

## Step 22 — Create three illustrative new inputs

Create three perturbed examples based on held-out measurements. These are synthetic teaching examples, not patient-specific predictions or medical advice. They are kept within a small change of real measurements so they remain near the observed data.

In [ ]:
# Start with three held-out examples and apply small, explicit changes to selected features.
new_inputs = X_test_selected.iloc[:3].copy()

# Perturb a few selected numeric measurements by small amounts.
if len(new_inputs) < 3:
    raise ValueError("At least three test examples are needed to create three illustrative inputs.")

for row_idx, multiplier in zip(new_inputs.index, [0.98, 1.00, 1.02]):
    for col in new_inputs.columns[:min(3, len(new_inputs.columns))]:
        new_inputs.loc[row_idx, col] = new_inputs.loc[row_idx, col] * multiplier

display(new_inputs)
new_scaled = scaler.transform(new_inputs)
new_predictions = final_model.predict(new_scaled)
new_probabilities = final_model.predict_proba(new_scaled)[:, 1]

prediction_table = pd.DataFrame({
    "Example": ["Illustrative input 1", "Illustrative input 2", "Illustrative input 3"],
    "Predicted diagnosis": np.where(new_predictions == 1, "M (malignant)", "B (benign)"),
    "Estimated malignant probability": new_probabilities.round(3)
})
display(prediction_table)

## Step 23 — Explain the results

Use the final outputs above to complete the short interpretation below. The metric values are generated from the current dataset when you run the notebook.

In [ ]:
print(f"Final accuracy: {final_metrics['Accuracy']:.3f}")
print(f"Malignant precision: {final_metrics['Precision (malignant)']:.3f}")
print(f"Malignant recall: {final_metrics['Malignant Recall']:.3f}")
print(f"Specificity: {final_metrics['Specificity (benign)']:.3f}")
print(f"ROC-AUC: {final_metrics['AUC']:.3f}")
print(f"Confusion matrix counts: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

## Step 24 — Conclusion

After running the notebook, use the actual metrics printed in Step 23 to describe the model. Do not report numbers from an earlier run if the current results differ.

The Tumour Tagger project uses Logistic Regression to classify breast tumour records as benign or malignant based on cell-nucleus measurements. The dataset was cleaned, explored with charts, and prepared for modelling. A derived area-to-perimeter ratio was created, and SelectKBest was used to select input features.

The model was evaluated using accuracy, precision, malignant recall, F1-score, specificity, a confusion matrix, and ROC-AUC. Malignant recall is especially important because a false negative means a malignant case was classified as benign. The final setting was chosen by comparing malignant recall and AUC.

This project helped me practise data cleaning, exploratory data analysis, feature engineering, train-test splitting, feature selection, scaling, Logistic Regression, and model evaluation using Python and scikit-learn. The model is for educational analysis only and must not be used as a substitute for professional medical diagnosis.

## Final submission checklist

- [ ] Run every cell from top to bottom without errors.
- [ ] Confirm `DATA_FILE` matches the actual CSV filename.
- [ ] Review the data dictionary and update any descriptions if needed.
- [ ] Verify the five EDA insights against the displayed charts.
- [ ] Save the final model comparison table, bar chart, confusion matrix, and ROC curve for the PDF report.
- [ ] Use the current run's final metrics in the report and conclusion.
- [ ] Save this notebook as `H39_P21_Rushan_Shaikh.ipynb`.